In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("Ambiente configurado para Milestone 3.")

Ambiente configurado para Milestone 3.


In [2]:
# Gerando 1.000 amostras simulando logística de resíduos
np.random.seed(42)
n = 1000

data = {
    'distancia_km': np.random.uniform(10, 600, n),
    'peso_carga_kg': np.random.uniform(200, 8000, n),
    'tipo_veiculo': np.random.choice(['Eletrico', 'Diesel', 'Hibrido', 'GNV'], n),
    'eficiencia_rota': np.random.uniform(0.6, 1.0, n), # 1.0 = rota otimizada
    'idade_veiculo_anos': np.random.randint(0, 15, n)
}

df = pd.DataFrame(data)

# Regra de negócio para gerar o target (Emissão CO2)
fator_emissao = {'Diesel': 2.65, 'Hibrido': 1.15, 'GNV': 1.85, 'Eletrico': 0.05}
df['target_co2'] = (df['distancia_km'] * (df['peso_carga_kg']/1000) * df['tipo_veiculo'].map(fator_emissao) / df['eficiencia_rota'])
df['target_co2'] += np.random.normal(0, 8, n) # Adicionando ruído real
df['target_co2'] = df['target_co2'].clip(lower=0) # Evitar valores negativos

print("Dataset de Economia Circular gerado com sucesso.")

Dataset de Economia Circular gerado com sucesso.


In [3]:
# Separação de features e target
X = df.drop('target_co2', axis=1)
y = df['target_co2']

# Divisão rigorosa Treino e Teste (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Definindo transformações
features_numericas = ['distancia_km', 'peso_carga_kg', 'eficiencia_rota', 'idade_veiculo_anos']
features_categoricas = ['tipo_veiculo']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), features_numericas),
        ('cat', OneHotEncoder(), features_categoricas)
    ])

print("Pré-processamento configurado e dados divididos (80/20).")

Pré-processamento configurado e dados divididos (80/20).


In [4]:
# 1. Regressão Linear (Baseline)
pipe_lr = Pipeline(steps=[('pre', preprocessor), ('model', LinearRegression())])
start_lr = time.time()
pipe_lr.fit(X_train, y_train)
time_lr = (time.time() - start_lr) * 1000

# 2. Random Forest Regressor (Complexo)
pipe_rf = Pipeline(steps=[('pre', preprocessor), ('model', RandomForestRegressor(n_estimators=100, random_state=42))])
start_rf = time.time()
pipe_rf.fit(X_train, y_train)
time_rf = (time.time() - start_rf) * 1000

print("Modelos treinados com sucesso.")

Modelos treinados com sucesso.


In [5]:
def calcular_metricas(model, X, y):
    preds = model.predict(X)
    return {
        'RMSE': np.sqrt(mean_squared_error(y, preds)),
        'MAE': mean_absolute_error(y, preds),
        'R2': r2_score(y, preds)
    }

metrics_lr = calcular_metricas(pipe_lr, X_test, y_test)
metrics_rf = calcular_metricas(pipe_rf, X_test, y_test)

# Tabela Comparativa (Template para o Notebook)
resultados = pd.DataFrame({
    'Métrica': ['RMSE', 'MAE', 'R2', 'Tempo Proc. (ms)'],
    'Linear Regression': [metrics_lr['RMSE'], metrics_lr['MAE'], metrics_lr['R2'], time_lr],
    'Random Forest': [metrics_rf['RMSE'], metrics_rf['MAE'], metrics_rf['R2'], time_rf]
})

print("\n--- RELATÓRIO DE PERFORMANCE ---")
print(resultados.to_markdown(index=False))

# Exportação do Modelo Escolhido (Pasta /models)
joblib.dump(pipe_rf, 'modelo_final.joblib')
print("\nModelo Final (Random Forest) exportado como 'modelo_final.joblib'.")


--- RELATÓRIO DE PERFORMANCE ---
| Métrica          |   Linear Regression |   Random Forest |
|:-----------------|--------------------:|----------------:|
| RMSE             |         1530.96     |      614.913    |
| MAE              |         1097.67     |      292.25     |
| R2               |            0.688156 |        0.949692 |
| Tempo Proc. (ms) |           40.5321   |      468.696    |

Modelo Final (Random Forest) exportado como 'modelo_final.joblib'.
